## 02 — Evaluation (base vs fine-tuned)

This notebook compares the **base** model vs **fine-tuned (LoRA)** on the held-out test split.

**Colab:** Runtime → Change runtime type → **GPU** (pick **T4** if listed) → Restart session → **Run all**.

- **Metrics:** BLEU, ROUGE-L, format compliance
- **Outputs:** `outputs/eval_metrics.json`, `outputs/qualitative_examples.md`

**New session?** The next cell can restore `lora_adapters` + `data/processed` from Google Drive if you saved them under `tatweer_challenge_submission`.


In [2]:
# Colab setup (GPU required for Unsloth)
import os
os.environ["WANDB_DISABLED"] = "true"

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. In Colab: Runtime → Change runtime type → Hardware accelerator: GPU. "
        "Then Runtime → Restart session, and re-run this cell."
    )

# Optional if CPU-only torch:
# !pip -q install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install -U unsloth transformers datasets accelerate peft trl bitsandbytes evaluate sacrebleu rouge-score scikit-learn matplotlib seaborn "pandas<3"

In [3]:
# Repo bootstrap — Colab or local Jupyter (same as notebook 01)
import os
import subprocess
from pathlib import Path
from typing import Optional

def _find_repo_dir() -> Optional[Path]:
    p = Path.cwd().resolve()
    for _ in range(16):
        if (p / "data" / "processed").is_dir() and (p / "scripts" / "generate_dataset.py").is_file():
            return p
        p = p.parent
    c = Path("/content/tatweer_challenge")
    return c if c.is_dir() else None

repo_dir = _find_repo_dir()
if repo_dir is None and Path("/content").is_dir():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/Shamem-cyberx/tatweer_challenge.git", "/content/tatweer_challenge"],
        check=False,
    )
    repo_dir = _find_repo_dir()
if repo_dir is None:
    raise FileNotFoundError(
        "Could not find tatweer_challenge. Open this notebook from .../tatweer_challenge/notebooks/ "
        "or clone to /content/tatweer_challenge (Colab)."
    )
os.chdir(str(repo_dir / "notebooks"))
print("repo_dir:", repo_dir)
print("cwd:", os.getcwd())

In [ ]:
# Optional: restore training artifacts from Google Drive (new Colab session wipes /content)
# Save path from notebook 01 copy: MyDrive/tatweer_challenge_submission
from pathlib import Path
import shutil

REPO = Path("/content/tatweer_challenge")
ADAPTER_MARKER = REPO / "outputs" / "lora_adapters" / "adapter_config.json"
TEST_JSONL = REPO / "data" / "processed" / "test.jsonl"

if not ADAPTER_MARKER.exists() or not TEST_JSONL.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        print("Not on Colab or Drive mount failed — ensure you ran notebook 01 in this session or copy files manually.")
    else:
        SRC = Path("/content/drive/MyDrive/tatweer_challenge_submission")
        if (SRC / "outputs" / "lora_adapters").exists():
            dst = REPO / "outputs" / "lora_adapters"
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(SRC / "outputs" / "lora_adapters", dst)
            print("Restored:", dst)
        if (SRC / "data" / "processed").exists():
            (REPO / "data" / "processed").mkdir(parents=True, exist_ok=True)
            for f in (SRC / "data" / "processed").glob("*.jsonl"):
                shutil.copy2(f, REPO / "data" / "processed" / f.name)
            print("Restored data/processed JSONL")
else:
    print("Artifacts already present — skip Drive restore.")

In [ ]:
# Load test set
from pathlib import Path
from datasets import load_dataset

root = Path("..").resolve()
data_dir = root / "data" / "processed"

ds = load_dataset(
    "json",
    data_files={"test": str(data_dir / "test.jsonl")},
)
test_ds = ds["test"]
len(test_ds), test_ds[0].keys()

In [5]:
# Load base model and fine-tuned adapters
def _ensure_unsloth():
    try:
        import unsloth  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    from IPython import get_ipython
    ip = get_ipython()
    pip_line = (
        'install -q -U unsloth transformers datasets accelerate peft trl bitsandbytes '
        'evaluate sacrebleu rouge-score scikit-learn matplotlib seaborn "pandas<3"'
    )
    if ip is not None:
        ip.run_line_magic("pip", pip_line)
    else:
        import subprocess
        import sys
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "--break-system-packages",
             "unsloth", "transformers", "datasets", "accelerate", "peft", "trl", "bitsandbytes",
             "evaluate", "sacrebleu", "rouge-score", "scikit-learn", "matplotlib", "seaborn", "pandas<3"],
        )

_ensure_unsloth()

import json
import torch
from pathlib import Path
from unsloth import FastLanguageModel
from peft import PeftModel

adapter_dir = (Path("..").resolve() / "outputs" / "lora_adapters")
if not (adapter_dir / "adapter_config.json").exists():
    raise FileNotFoundError(
        f"Missing LoRA adapters at {adapter_dir}. Run notebook 01 first or run the Drive restore cell above."
    )
default_name = "Qwen/Qwen2.5-0.5B-Instruct"
cfg_path = adapter_dir / "adapter_config.json"
model_name = json.loads(cfg_path.read_text(encoding="utf-8")).get(
    "base_model_name_or_path", default_name
)
# Must match the checkpoint LoRA was trained on (see adapter_config.json). Wrong base → empty FT outputs.
print("Checkpoint:", model_name)

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

ft_model, _ = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

ft_model = PeftModel.from_pretrained(ft_model, str(adapter_dir))

base_model.eval()
ft_model.eval()
FastLanguageModel.for_inference(base_model)
FastLanguageModel.for_inference(ft_model)

print("Adapter dir:", adapter_dir)
print("CUDA:", torch.cuda.is_available())

In [6]:
# Inference helper (greedy decode; pad_token avoids warnings)
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")


def generate(model, instruction: str, max_new_tokens: int = 256) -> str:
    prompt = "\n".join([
        "### Instruction:",
        instruction.strip(),
        "",
        "### Response:",
    ])
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    eos_id = tokenizer.eos_token_id
    gen_kw = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=pad_id,
    )
    if eos_id is not None:
        gen_kw["eos_token_id"] = eos_id
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kw)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip()

# Quick sanity check
print("BASE:\n", generate(base_model, test_ds[0]["instruction"], 192)[:500])
print("\nFT:\n", generate(ft_model, test_ds[0]["instruction"], 192)[:500])

In [ ]:
# Quantitative evaluation
import sacrebleu
from rouge_score import rouge_scorer
import json

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def format_compliance(pred: str, expects_code: bool) -> int:
    if not expects_code:
        return 1
    # expects code: require a fenced code block and at least one PowerShell-ish token
    has_fence = "```" in pred
    has_cmd = any(tok in pred for tok in ["Get-", "Set-", "Test-", "Restart-", "Resolve-", "ipconfig", "sc.exe", "tracert"])
    return int(has_fence and has_cmd)


def run_eval(model):
    preds, refs = [], []
    fmt = []
    for ex in test_ds:
        p = generate(model, ex["instruction"], max_new_tokens=256)
        preds.append(p)
        refs.append(ex["response"].strip())
        fmt.append(format_compliance(p, bool(ex.get("expects_code", False))))

    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    rougeL = sum(rouge.score(r, p)["rougeL"].fmeasure for r, p in zip(refs, preds)) / len(preds)
    fmt_acc = sum(fmt) / len(fmt)

    return {
        "bleu": float(bleu),
        "rougeL_f": float(rougeL),
        "format_compliance": float(fmt_acc),
        "preds": preds,
        "refs": refs,
    }

base_res = run_eval(base_model)
ft_res = run_eval(ft_model)

metrics = {
    "base": {k: v for k, v in base_res.items() if k in ["bleu", "rougeL_f", "format_compliance"]},
    "fine_tuned": {k: v for k, v in ft_res.items() if k in ["bleu", "rougeL_f", "format_compliance"]},
}
metrics

In [ ]:
# Save outputs + qualitative examples (Challenge 2 — structured eval log)
from pathlib import Path
from datetime import datetime, timezone

out_dir = Path("..").resolve() / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

_wpath = out_dir / "lora_adapters" / "adapter_model.safetensors"
_weights_ok = _wpath.is_file() and _wpath.stat().st_size > 0
_warnings = []
if not _weights_ok:
    _warnings.append(
        "adapter_model.safetensors missing or empty — fine_tuned metrics may be invalid; "
        "run 01_dataset_and_train.ipynb on GPU, then re-run this notebook."
    )

eval_payload = {
    "meta": {
        "challenge": "Challenge 2",
        "test_jsonl": str(data_dir / "test.jsonl"),
        "n_test_examples": len(test_ds),
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
        "adapter_dir": str(out_dir / "lora_adapters"),
        "adapter_weights_present": _weights_ok,
    },
    "warnings": _warnings,
    "base": metrics["base"],
    "fine_tuned": metrics["fine_tuned"],
}

with open(out_dir / "eval_metrics.json", "w", encoding="utf-8") as f:
    json.dump(eval_payload, f, ensure_ascii=False, indent=2)

# 8 examples: mix good/bad by comparing rougeL per example
scores = []
for i, (ref, p_base, p_ft) in enumerate(zip(base_res["refs"], base_res["preds"], ft_res["preds"])):
    s_base = rouge.score(ref, p_base)["rougeL"].fmeasure
    s_ft = rouge.score(ref, p_ft)["rougeL"].fmeasure
    scores.append((i, s_base, s_ft))

# pick top improvements and worst regressions
scores_sorted = sorted(scores, key=lambda x: (x[2] - x[1]), reverse=True)
show_ids = [x[0] for x in scores_sorted[:4]] + [x[0] for x in scores_sorted[-4:]]

md_lines = ["## Qualitative examples (base vs fine-tuned)", ""]
for i in show_ids:
    ex = test_ds[i]
    md_lines += [
        f"### Example {i} — {ex.get('category','')}",
        "**Instruction**:",
        ex["instruction"],
        "",
        "**Reference**:",
        ex["response"],
        "",
        "**Base output**:",
        base_res["preds"][i],
        "",
        "**Fine-tuned output**:",
        ft_res["preds"][i],
        "",
        "---",
        "",
    ]

(out_dir / "qualitative_examples.md").write_text("\n".join(md_lines), encoding="utf-8")

print("Saved:", out_dir / "eval_metrics.json")
print("Saved:", out_dir / "qualitative_examples.md")
metrics

### Challenge 2 — what this notebook must show (assessment)

| Requirement | Where |
|-------------|--------|
| ≥2 task metrics + base vs FT | Corpus **BLEU**, **ROUGE-L**, **format compliance** |
| Loss curves (from training) | Optional cell loads `outputs/trainer_log_history.json` from notebook **01** |
| Qualitative + error analysis | `qualitative_examples.md` + per-example ROUGE table |
| Honest limitations | If metrics look odd, use the **diagnostics** cell |

**Saved files (notebook 01 → `outputs/`):** `lora_adapters/`, `loss_curve.png`, `trainer_log_history.json`, `training_summary.json`, `checkpoints/`  
**Saved files (notebook 02 → `outputs/`):** `eval_metrics.json` (includes `meta` + `warnings`), `qualitative_examples.md`, `eval_per_example.json`, `eval_metrics_bar.png`, `eval_rouge_delta_hist.png`, `challenge2_output_manifest.json`, optional `eval_loss_from_trainer_log.png`, optional `demo_query_snapshot.*`.

In [ ]:
# Artifact checks — confirm submission files exist before trusting metrics
import json
from pathlib import Path

root = Path("..").resolve()
out_dir = root / "outputs"
adapt = out_dir / "lora_adapters"

need = [
    adapt / "adapter_config.json",
    adapt / "adapter_model.safetensors",
    out_dir / "eval_metrics.json",
]
def _human_bytes(n: int) -> str:
    if n >= 1_048_576:
        return f"{n / 1_048_576:.2f} MB"
    if n >= 1024:
        return f"{n / 1024:.1f} KB"
    return f"{n} B"

for p in need:
    ok = p.is_file() and p.stat().st_size > 0
    sz = _human_bytes(p.stat().st_size) if ok else "MISSING or empty"
    print(f"{'OK ' if ok else '!! '}{p.name}: {sz}")

if not (adapt / "adapter_model.safetensors").is_file():
    print(
        "\nWARNING: No adapter weights — PeftModel may be uninitialized; "
        "fine-tuned metrics can be meaningless. Re-run notebook 01 and copy adapter_model.safetensors."
    )

# Write a single manifest JSON for Challenge 2 submission checks
manifest_paths = [
    "lora_adapters/adapter_config.json",
    "lora_adapters/adapter_model.safetensors",
    "training_summary.json",
    "trainer_log_history.json",
    "loss_curve.png",
    "eval_metrics.json",
    "qualitative_examples.md",
    "eval_per_example.json",
    "eval_metrics_bar.png",
    "eval_rouge_delta_hist.png",
    "eval_loss_from_trainer_log.png",
]
manifest = {"challenge": "Challenge 2", "outputs_dir": str(out_dir), "files": {}}
for rel in manifest_paths:
    p = out_dir / rel
    ok = p.is_file() and p.stat().st_size > 0
    manifest["files"][rel] = {
        "exists": ok,
        "size_bytes": p.stat().st_size if ok else 0,
    }
mp = out_dir / "challenge2_output_manifest.json"
mp.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", mp)

In [ ]:
# Charts + per-example JSON (for reports / overfitting discussion)
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

out_dir = Path("..").resolve() / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

rows = []
for i in range(len(base_res["refs"])):
    ref, pb, pf = base_res["refs"][i], base_res["preds"][i], ft_res["preds"][i]
    rb = rouge.score(ref, pb)["rougeL"].fmeasure
    rf = rouge.score(ref, pf)["rougeL"].fmeasure
    rows.append({
        "idx": i,
        "category": test_ds[i].get("category", ""),
        "rougeL_base": float(rb),
        "rougeL_finetuned": float(rf),
        "delta_rougeL": float(rf - rb),
        "pred_chars_base": len(pb),
        "pred_chars_ft": len(pf),
    })

with open(out_dir / "eval_per_example.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)

names = ["BLEU", "ROUGE-L", "Format"]
b_vals = [metrics["base"]["bleu"], metrics["base"]["rougeL_f"], metrics["base"]["format_compliance"]]
f_vals = [metrics["fine_tuned"]["bleu"], metrics["fine_tuned"]["rougeL_f"], metrics["fine_tuned"]["format_compliance"]]

x = np.arange(len(names))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w / 2, b_vals, w, label="base", color="steelblue")
ax.bar(x + w / 2, f_vals, w, label="fine_tuned", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel("score")
ax.set_title("Test set — corpus metrics (base vs fine-tuned)")
ax.legend()
ax.set_ylim(bottom=0)
plt.tight_layout()
fig.savefig(out_dir / "eval_metrics_bar.png", dpi=160)
plt.show()

deltas = [r["delta_rougeL"] for r in rows]
fig2, ax2 = plt.subplots(figsize=(9, 3.5))
ax2.hist(deltas, bins=min(12, len(deltas)), color="steelblue", edgecolor="white")
ax2.axvline(0.0, color="crimson", linestyle="--", linewidth=1, label="no change")
ax2.set_xlabel("ROUGE-L(FT) − ROUGE-L(base) per example")
ax2.set_ylabel("count")
ax2.set_title("Distribution of per-example improvement (ROUGE-L)")
ax2.legend()
plt.tight_layout()
fig2.savefig(out_dir / "eval_rouge_delta_hist.png", dpi=160)
plt.show()

print("Saved:", out_dir / "eval_per_example.json")
print("Saved:", out_dir / "eval_metrics_bar.png", out_dir / "eval_rouge_delta_hist.png")

In [ ]:
# Training loss curves (from notebook 01) — satisfies "loss curves" in the brief if file exists
import json
from pathlib import Path
import matplotlib.pyplot as plt

hist_path = Path("..").resolve() / "outputs" / "trainer_log_history.json"
if not hist_path.is_file():
    print("No trainer_log_history.json — run 01_dataset_and_train.ipynb to produce it, or copy outputs from Colab.")
else:
    hist = json.loads(hist_path.read_text(encoding="utf-8"))
    tr_s, tr_l, ev_s, ev_l = [], [], [], []
    for row in hist:
        if "eval_loss" in row and "step" in row:
            ev_s.append(row["step"])
            ev_l.append(row["eval_loss"])
        elif "loss" in row and "step" in row:
            tr_s.append(row["step"])
            tr_l.append(row["loss"])
    fig, ax = plt.subplots(figsize=(9, 4))
    if tr_s:
        ax.plot(tr_s, tr_l, label="train", alpha=0.85)
    if ev_s:
        ax.plot(ev_s, ev_l, label="validation", linewidth=2)
    ax.set_xlabel("step")
    ax.set_ylabel("loss")
    ax.set_title("Train / validation loss (from trainer log)")
    ax.legend()
    plt.tight_layout()
    outp = Path("..").resolve() / "outputs" / "eval_loss_from_trainer_log.png"
    fig.savefig(outp, dpi=160)
    plt.show()
    print("Saved:", outp)

### How to read results (accuracy & sanity)

- **Corpus BLEU / ROUGE-L**: Higher usually means closer overlap to reference answers; small data + diverse phrasing keeps scores modest.
- **Format compliance**: Share of answers that include a fenced code block and a cmdlet-like token when `expects_code` is true.
- **All fine-tuned metrics exactly 0.0**: Often means **empty generations**, **wrong adapter load**, or a **metric bug** — use the next cell.
- **Per-example `delta_rougeL`**: Positive means FT matched the reference better than base on that item; a left-heavy histogram with many negatives suggests regression or overfitting.
- **Loss plot**: Validation loss flat or rising while train drops → possible overfitting (mention in write-up).

In [ ]:
# Diagnostics — explain zeros / odd corpus scores
lens_ft = [len(p) for p in ft_res["preds"]]
lens_b = [len(p) for p in base_res["preds"]]
empty_ft = sum(1 for p in ft_res["preds"] if not p.strip())
empty_b = sum(1 for p in base_res["preds"] if not p.strip())

print("Prediction length (chars) — base: min/mean/max", min(lens_b), sum(lens_b) / len(lens_b), max(lens_b))
print("Prediction length (chars) — FT:  min/mean/max", min(lens_ft), sum(lens_ft) / len(lens_ft), max(lens_ft))
print("Empty predictions — base:", empty_b, " FT:", empty_ft)

z = metrics["fine_tuned"]
if z["bleu"] == 0 and z["rougeL_f"] == 0 and z["format_compliance"] == 0:
    print(
        "\n*** Fine-tuned corpus metrics are all zero. Typical causes: ***\n"
        "  1) adapter_model.safetensors missing or not loaded\n"
        "  2) All FT predictions empty or degenerate\n"
        "  3) Re-run evaluation after fixing adapters\n"
    )

print("\nFirst test example — FT preview:\n", ft_res["preds"][0][:400].replace("\n", " ")[:400])

### Batch snapshot (saved files)

Run the next cell after models + `generate()` exist. It runs **fixed demo queries** (no `input()`), compares **BASE vs FINE-TUNED**, and saves:

- `outputs/demo_query_snapshot.md` — human-readable
- `outputs/demo_query_snapshot.json` — full text for tooling
- `outputs/demo_query_snapshot.html` — open in browser or screenshot for reports

Set `RUN_BATCH_QUERY_SNAPSHOT = False` to skip during **Run all** (saves time).

In [ ]:
# Fixed queries → BASE vs FT → save snapshot files (no input(); needs GPU + models in memory)
import json
import logging
from datetime import datetime, timezone
from html import escape
from pathlib import Path

_LOG_SN = logging.getLogger("transformers")

# Set False to skip (e.g. fast Run all without re-generating 10 long answers)
RUN_BATCH_QUERY_SNAPSHOT = True

FIXED_DEMO_QUERIES = [
    "How do I see what's using a lot of CPU right now? Keep it simple.",
    "I need to list running services and maybe restart one — what commands should I try?",
    "I'm seeing this error in PowerShell: 'Access is denied'. Explain likely causes and give a short diagnostic checklist with commands.",
    "My script won't run; something about execution policy. How do I check without breaking stuff?",
    "Invoke-WebRequest fails with SSL/TLS connection error. What should I check first?",
]

_MAX_SNAP = 4000


def _gen_snap(model, q: str) -> str:
    p = _LOG_SN.level
    _LOG_SN.setLevel(logging.ERROR)
    try:
        return generate(model, q, max_new_tokens=256)
    finally:
        _LOG_SN.setLevel(p)


def _md_indent(s: str, max_len: int) -> str:
    body = s[:max_len] + ("…" if len(s) > max_len else "")
    return "\n".join("    " + line for line in body.splitlines())


if RUN_BATCH_QUERY_SNAPSHOT:
    _need = ("base_model", "ft_model", "generate")
    if any(x not in globals() for x in _need):
        print("Skip batch snapshot — run load-model + generate cells first.")
    else:
        out_dir = Path("..").resolve() / "outputs"
        out_dir.mkdir(parents=True, exist_ok=True)
        ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        rows = []
        md_lines = [
            "# Base vs fine-tuned — batch query snapshot",
            "",
            f"*UTC: {ts}*",
            "",
        ]
        html_parts = [
            "<!DOCTYPE html><html><head><meta charset='utf-8'><title>demo_query_snapshot</title>",
            "<style>body{font-family:system-ui,max-width:960px;margin:24px auto;padding:0 16px;}",
            "h2{border-bottom:1px solid #ccc;padding-bottom:6px;} .q{background:#fffbe6;padding:12px;border-radius:8px;}",
            "pre{white-space:pre-wrap;background:#f6f8fa;border:1px solid #d0d7de;padding:12px;border-radius:8px;}",
            ".b{border-left:4px solid #2563eb;} .f{border-left:4px solid #ea580c;}</style></head><body>",
            f"<h1>Base vs FT — snapshot</h1><p><small>{escape(ts)}</small></p>",
        ]

        for i, q in enumerate(FIXED_DEMO_QUERIES, 1):
            b = _gen_snap(base_model, q)
            f = _gen_snap(ft_model, q)
            rows.append(
                {
                    "id": i,
                    "query": q,
                    "base_chars": len(b),
                    "ft_chars": len(f),
                    "base": b,
                    "fine_tuned": f,
                }
            )
            md_lines += [
                f"## Query {i}",
                "",
                "**Instruction:** " + q.replace("\n", " "),
                "",
                f"**BASE** ({len(b)} chars)",
                "",
                _md_indent(b, _MAX_SNAP),
                "",
                f"**FINE-TUNED** ({len(f)} chars)",
                "",
                _md_indent(f, _MAX_SNAP),
                "",
                "---",
                "",
            ]
            html_parts += [
                f"<h2>Query {i}</h2><div class='q'><strong>Instruction</strong><pre>{escape(q)}</pre></div>",
                "<h3 class='b'>BASE</h3><pre>", escape(b[:_MAX_SNAP] + ("…" if len(b) > _MAX_SNAP else "")), "</pre>",
                "<h3 class='f'>FINE-TUNED</h3><pre>", escape(f[:_MAX_SNAP] + ("…" if len(f) > _MAX_SNAP else "")), "</pre><hr>",
            ]

        html_parts.append("</body></html>")
        (out_dir / "demo_query_snapshot.md").write_text("\n".join(md_lines), encoding="utf-8")
        (out_dir / "demo_query_snapshot.json").write_text(
            json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        (out_dir / "demo_query_snapshot.html").write_text("".join(html_parts), encoding="utf-8")
        print("Saved:", out_dir / "demo_query_snapshot.md")
        print("Saved:", out_dir / "demo_query_snapshot.json")
        print("Saved:", out_dir / "demo_query_snapshot.html")
else:
    print("Batch snapshot skipped (RUN_BATCH_QUERY_SNAPSHOT = False).")

In [ ]:
# =============================================================================
# Interviewer demo — ONE prompt → BASE + FINE-TUNED (same question)
# =============================================================================
# Run all cells above. Set RUN_INTERACTIVE_DEMO = True, run this cell only.
# =============================================================================

import logging
from html import escape

try:
    from IPython.display import display, HTML
except ImportError:
    display = lambda x: None  # noqa: E731
    HTML = str

RUN_INTERACTIVE_DEMO = True  # True for live demo; False so Run All does not block

_DEMO_MAX_CHARS = 1200
_LOG = logging.getLogger("transformers")

_CSS = """
<style>
.demo-wrap { max-width: 980px; margin: 0 auto; font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; }
.demo-query {
  background: linear-gradient(180deg, #fffbeb 0%, #fff7d6 100%);
  border: 1px solid #f5d565; border-radius: 12px; padding: 14px 16px; margin: 12px 0 18px;
  box-shadow: 0 1px 2px rgba(15, 23, 42, 0.06);
}
.demo-query strong { font-size: 13px; color: #854d0e; letter-spacing: 0.02em; }
.demo-query pre { margin: 10px 0 0; white-space: pre-wrap; font-size: 14px; line-height: 1.5; color: #1e293b; }

.demo-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 14px; }
@media (max-width: 900px) { .demo-grid { grid-template-columns: 1fr; } }

.demo-card {
  border-radius: 12px; overflow: hidden; border: 1px solid #e2e8f0;
  box-shadow: 0 2px 8px rgba(15, 23, 42, 0.06);
  background: #fff;
}
.demo-card-h {
  padding: 10px 14px; font-size: 13px; font-weight: 650; letter-spacing: 0.02em;
  color: #0f172a; border-bottom: 1px solid #e2e8f0;
}
.demo-card-h.base { background: #eff6ff; border-left: 4px solid #2563eb; }
.demo-card-h.ft   { background: #fff7ed; border-left: 4px solid #ea580c; }

.demo-card pre {
  margin: 0; padding: 14px 14px 16px; white-space: pre-wrap; font-size: 13px; line-height: 1.5;
  color: #334155; background: #f8fafc; max-height: 420px; overflow: auto;
}
.demo-foot {
  margin: 12px 0 22px; font-size: 12px; color: #64748b;
  padding: 8px 2px; border-top: 1px solid #e2e8f0;
}
</style>
"""


def _gen_quiet(model, instruction: str) -> str:
    prev = _LOG.level
    _LOG.setLevel(logging.ERROR)
    try:
        return generate(model, instruction, max_new_tokens=256)
    finally:
        _LOG.setLevel(prev)


def _show_round(n: int, q: str, base_text: str, ft_text: str) -> None:
    b = base_text[:_DEMO_MAX_CHARS] + ("…" if len(base_text) > _DEMO_MAX_CHARS else "")
    f = ft_text[:_DEMO_MAX_CHARS] + ("…" if len(ft_text) > _DEMO_MAX_CHARS else "")

    html = f"""
{_CSS}
<div class="demo-wrap">
  <div class="demo-query">
    <strong>QUERY #{n}</strong>
    <pre>{escape(q)}</pre>
  </div>

  <div class="demo-grid">
    <div class="demo-card">
      <div class="demo-card-h base">BASE · pretrained (no LoRA)</div>
      <pre>{escape(b)}</pre>
    </div>
    <div class="demo-card">
      <div class="demo-card-h ft">FINE-TUNED · QLoRA</div>
      <pre>{escape(f)}</pre>
    </div>
  </div>

  <div class="demo-foot">
    Length (chars) — base: {len(base_text)} · fine-tuned: {len(ft_text)}
  </div>
</div>
"""
    display(HTML(html))


def _interviewer_demo() -> None:
    need = ("base_model", "ft_model", "generate")
    missing = [n for n in need if n not in globals()]
    if missing:
        print("Missing:", ", ".join(missing), "— run all cells above first.")
        return

    print("One instruction → side-by-side BASE vs FINE-TUNED. Empty line or quit = exit.\n")
    n = 0
    while True:
        try:
            line = input("▸ Instruction: ").strip()
        except EOFError:
            print("(EOF)")
            break
        if not line or line.lower() in ("quit", "exit", "q"):
            print("Done.")
            break

        n += 1
        base_text = _gen_quiet(base_model, line)
        ft_text = _gen_quiet(ft_model, line)
        _show_round(n, line, base_text, ft_text)


if RUN_INTERACTIVE_DEMO:
    _interviewer_demo()
else:
    print(
        "Interactive demo skipped (RUN_INTERACTIVE_DEMO = False). "
        "Set True and run this cell — one prompt shows both models side by side."
    )

In [ ]:
# GPU sanity check — same as notebook 01 last cell (run anytime; confirms this kernel sees T4)
import sys

import torch

print("Python exe:", sys.executable)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("device 0:", torch.cuda.get_device_name(0))
    print("device count:", torch.cuda.device_count())
else:
    print(
        "\nNo CUDA in this kernel — Unsloth/QLoRA needs a GPU in this session.\n"
        "If you expect Colab T4:\n"
        "  • Runtime → Change runtime type → GPU (T4) → Save, then Runtime → Restart session.\n"
        "  • In VS Code/Cursor: pick the Colab/Jupyter kernel that runs on Colab, not local Windows Python.\n"
        "  • A CPU-only torch build (often shows '+cpu' in torch.__version__) cannot see a GPU.\n"
    )
